# Tools

The five read-only tools the agent (and later the scripted baseline) can call. Each is a
plain function of the snapshot, the review day and its typed arguments, and returns
facts: counts, averages, model labels, rows. No tool says what a fact means, names a
planted condition, or suggests the next step; reading the facts is the agent's job.

Each tool ignores anything after the review day. A request a tool cannot answer, such
as an unknown series, raises `ValueError`, which the gateway turns into a failed reply.

Unless a model version is named, tools judge the forecasting model under review (every
model except the `seasonal_naive_v1` benchmark), whatever version label it carries.

In [ ]:
#| default_exp tools

In [ ]:
#| export
from collections.abc import Iterable

from agentic_forecast_review.gateway import (
    Tool,
    ToolOutput,
)
from agentic_forecast_review.generator import (
    ActualRow,
    ForecastRow,
    Snapshot,
    actual_row_id,
)
from agentic_forecast_review.metrics import (
    mae,
    signed_bias,
)
from agentic_forecast_review.schemas import (
    CheckQualityArgs,
    CompareRunsArgs,
    DayRange,
    EvidenceRowsArgs,
    FieldValue,
    InspectSeriesArgs,
    Metric,
    ResolveContextArgs,
    ToolName,
)

## Shared helpers

Pick one series' actuals and forecasts inside a day range; spell a set of horizons as
compact spans (`1-7,22-28`). Numbers are rounded to 3 decimals.

In [ ]:
#| export
BENCHMARK_MODEL = "seasonal_naive_v1"
MAX_EVIDENCE_ROWS = 50


def _check_series(snapshot: Snapshot, series_ids: Iterable[str]) -> None:
    known = {row.series_id for row in snapshot.actuals}
    unknown = sorted(set(series_ids) - known)
    if unknown:
        raise ValueError(f"unknown series: {', '.join(unknown)}")


def _days(days: DayRange, review_day: int) -> range:
    return range(days.start_day, min(days.end_day, review_day) + 1)


def _actuals(
    snapshot: Snapshot, series_id: str, days: DayRange, review_day: int
) -> list[ActualRow]:
    wanted = _days(days, review_day)
    return [
        row
        for row in snapshot.actuals
        if row.series_id == series_id and row.day in wanted
    ]


def _forecasts(
    snapshot: Snapshot,
    series_id: str,
    days: DayRange,
    review_day: int,
    model_version: str | None = None,
) -> list[ForecastRow]:
    wanted = _days(days, review_day)
    return [
        row
        for row in snapshot.forecasts
        if row.series_id == series_id
        and row.target_day in wanted
        and (
            row.model_version == model_version
            if model_version
            else row.model_version != BENCHMARK_MODEL
        )
    ]


def _first_by_day(rows: Iterable[ActualRow]) -> dict[int, int]:
    by_day: dict[int, int] = {}
    for row in rows:
        by_day.setdefault(row.day, row.value)
    return by_day


def _spans(values: Iterable[int]) -> str:
    spans: list[list[int]] = []
    for value in sorted(set(values)):
        if spans and value == spans[-1][1] + 1:
            spans[-1][1] = value
        else:
            spans.append([value, value])
    return ",".join(f"{a}-{b}" if a != b else str(a) for a, b in spans) or "none"

## 1. resolve_context

What the snapshot holds: its days, series, model versions and the metrics a request
may use. The table lists every series. This is also where a vague request finds its
options (`metrics`).

In [ ]:
#| export
def resolve_context(
    snapshot: Snapshot, review_day: int, args: ResolveContextArgs
) -> ToolOutput:
    """The snapshot's days, series, model versions and allowed metrics."""
    days = [row.day for row in snapshot.actuals if row.day <= review_day]
    series = sorted({row.series_id for row in snapshot.actuals})
    return ToolOutput(
        fields={
            "snapshot_id": snapshot.snapshot_id,
            "first_day": min(days),
            "last_day": max(days),
            "review_day": review_day,
            "n_series": len(series),
            "model_versions": ",".join(
                sorted({row.model_version for row in snapshot.forecasts})
            ),
            "benchmark_model": BENCHMARK_MODEL,
            "issue_days": ",".join(
                str(day)
                for day in sorted({row.issue_day for row in snapshot.forecasts})
            ),
            "metrics": ",".join(metric.value for metric in Metric),
        },
        table=tuple({"series_id": series_id} for series_id in series),
    )

## 2. compare_runs

For each series: error (MAE) and bias in the review period and in the window before it,
the change between them, how many actual rows were found against the days expected, and
which model version made the forecasts. Fields are prefixed by the series, such as
`ca_2_foods__mae_period`.

Each forecast is matched with every actual row for its day, the way a plain join on the
day would; a day listed twice therefore counts twice.

In [ ]:
#| export
def compare_runs(
    snapshot: Snapshot, review_day: int, args: CompareRunsArgs
) -> ToolOutput:
    """MAE, bias, row counts and model labels: review period against the window."""
    _check_series(snapshot, args.series)
    fields: dict[str, FieldValue] = {}
    for series_id in args.series:
        prefix = series_id.lower()
        found: dict[str, tuple[float, float]] = {}
        for label, days in (("period", args.period), ("window", args.window)):
            actuals = _actuals(snapshot, series_id, days, review_day)
            observed: dict[int, list[int]] = {}
            for actual_row in actuals:
                observed.setdefault(actual_row.day, []).append(actual_row.value)
            forecasts = _forecasts(
                snapshot, series_id, days, review_day, args.model_version
            )
            pairs = [
                (value, row.value)
                for row in forecasts
                for value in observed.get(row.target_day, [])
            ]
            if not pairs:
                raise ValueError(
                    f"no forecasts with actuals for {series_id} in {label}"
                )
            actual = [pair[0] for pair in pairs]
            predicted = [pair[1] for pair in pairs]
            found[label] = (mae(actual, predicted), signed_bias(actual, predicted))
            fields[f"{prefix}__mae_{label}"] = round(found[label][0], 3)
            fields[f"{prefix}__bias_{label}"] = round(found[label][1], 3)
            fields[f"{prefix}__n_actual_{label}"] = len(actuals)
            fields[f"{prefix}__n_expected_{label}"] = len(_days(days, review_day))
            fields[f"{prefix}__model_{label}"] = ",".join(
                sorted({row.model_version for row in forecasts})
            )
        fields[f"{prefix}__mae_delta"] = round(
            found["period"][0] - found["window"][0], 3
        )
        fields[f"{prefix}__bias_delta"] = round(
            found["period"][1] - found["window"][1], 3
        )
    return ToolOutput(fields=fields)

## 3. inspect_series

One series day by day: actual, forecast, how far ahead it was made, and by which model.
The fields summarise it: averages, recent bias (last 7 and 14 days), and how many days
have both an actual and a forecast. The row IDs are the actuals shown.

In [ ]:
#| export
def inspect_series(
    snapshot: Snapshot, review_day: int, args: InspectSeriesArgs
) -> ToolOutput:
    """A day-by-day table of actuals and forecasts for one series."""
    _check_series(snapshot, [args.series_id])
    observed = _first_by_day(_actuals(snapshot, args.series_id, args.days, review_day))
    forecasts = {
        row.target_day: row
        for row in sorted(
            _forecasts(snapshot, args.series_id, args.days, review_day),
            key=lambda row: row.issue_day,
        )
    }
    days = [
        day
        for day in _days(args.days, review_day)
        if day in observed and day in forecasts
    ]
    if not days:
        raise ValueError(
            f"no days with both an actual and a forecast for {args.series_id}"
        )
    table: tuple[dict[str, FieldValue], ...] = tuple(
        {
            "day": day,
            "actual": observed[day],
            "forecast": forecasts[day].value,
            "horizon": forecasts[day].horizon,
            "model": forecasts[day].model_version,
        }
        for day in days
    )
    actual = [observed[day] for day in days]
    predicted = [forecasts[day].value for day in days]

    def bias_last(n: int) -> float:
        return round(signed_bias(actual[-n:], predicted[-n:]), 3)

    return ToolOutput(
        fields={
            "n_days": len(table),
            "mean_actual": round(sum(actual) / len(actual), 3),
            "mean_forecast": round(sum(predicted) / len(predicted), 3),
            "bias_last_7": bias_last(7),
            "bias_last_14": bias_last(14),
        },
        row_ids=tuple(actual_row_id(args.series_id, day) for day in days),
        table=table,
    )

## 4. check_quality

For each series, is the data fit to compare: days with no actual, actual rows listed
more than once, coverage, how many version labels the model's forecasts carry, and which
horizons (days ahead) they were made at. The row IDs are the affected rows: missing
actuals and duplicated ones.

In [ ]:
#| export
def check_quality(
    snapshot: Snapshot, review_day: int, args: CheckQualityArgs
) -> ToolOutput:
    """Missing and duplicate actuals, coverage, version labels and horizons."""
    _check_series(snapshot, args.series)
    fields: dict[str, FieldValue] = {}
    affected: list[str] = []
    wanted = _days(args.days, review_day)
    for series_id in args.series:
        prefix = series_id.lower()
        actuals = _actuals(snapshot, series_id, args.days, review_day)
        present = {row.day for row in actuals}
        missing = [day for day in wanted if day not in present]
        counts: dict[str, int] = {}
        for row in actuals:
            counts[row.row_id] = counts.get(row.row_id, 0) + 1
        forecasts = _forecasts(snapshot, series_id, args.days, review_day)
        fields[f"{prefix}__missing_days"] = len(missing)
        fields[f"{prefix}__duplicate_rows"] = len(actuals) - len(present)
        fields[f"{prefix}__coverage"] = round(len(present) / len(wanted), 3)
        fields[f"{prefix}__version_count"] = len(
            {row.model_version for row in forecasts}
        )
        fields[f"{prefix}__horizon_range"] = _spans(row.horizon for row in forecasts)
        affected += [actual_row_id(series_id, day) for day in missing]
        affected += [row_id for row_id, count in counts.items() if count > 1]
    return ToolOutput(fields=fields, row_ids=tuple(affected))

## 5. evidence_rows

The exact rows behind a finding, by row ID (at most 50 at a time). Actual and forecast
rows share one table; a column that does not apply is left empty or 0. `n_found` counts
the requested IDs that exist; a row listed twice in the data appears twice.

In [ ]:
#| export
def _actual_entry(row: ActualRow) -> dict[str, FieldValue]:
    return {
        "row_id": row.row_id,
        "kind": "actual",
        "series_id": row.series_id,
        "model": "",
        "issue_day": 0,
        "target_day": row.day,
        "horizon": 0,
        "value": row.value,
    }


def _forecast_entry(row: ForecastRow) -> dict[str, FieldValue]:
    return {
        "row_id": row.row_id,
        "kind": "forecast",
        "series_id": row.series_id,
        "model": row.model_version,
        "issue_day": row.issue_day,
        "target_day": row.target_day,
        "horizon": row.horizon,
        "value": row.value,
    }


def evidence_rows(
    snapshot: Snapshot, review_day: int, args: EvidenceRowsArgs
) -> ToolOutput:
    """The rows with the requested IDs, as a table."""
    wanted = list(dict.fromkeys(args.row_ids))
    if len(wanted) > MAX_EVIDENCE_ROWS:
        raise ValueError(f"at most {MAX_EVIDENCE_ROWS} row IDs per request")
    table: list[dict[str, FieldValue]] = []
    found: list[str] = []
    for row_id in wanted:
        rows = [
            _actual_entry(row)
            for row in snapshot.actuals
            if row.row_id == row_id and row.day <= review_day
        ]
        rows += [
            _forecast_entry(row)
            for row in snapshot.forecasts
            if row.row_id == row_id and row.issue_day <= review_day
        ]
        if rows:
            found.append(row_id)
            table += rows
    return ToolOutput(
        fields={"n_rows": len(wanted), "n_found": len(found)},
        row_ids=tuple(found),
        table=tuple(table),
    )

The tools by name, ready to hand to the gateway.

In [ ]:
#| export
TOOLS: dict[ToolName, Tool] = {
    ToolName.RESOLVE_CONTEXT: resolve_context,
    ToolName.COMPARE_RUNS: compare_runs,
    ToolName.INSPECT_SERIES: inspect_series,
    ToolName.CHECK_QUALITY: check_quality,
    ToolName.EVIDENCE_ROWS: evidence_rows,
}

## A hand-worked example

Two series, 21 days, small enough to check with pen and paper. One forecast is issued
on day 7 for days 8–21 (1–14 days ahead). The window is days 8–14, the review period
days 15–21, and the review day is 21.

- **X_1_FOODS** sells 10 every day. The model forecasts 11 in the window (1 too high
  each day) and 13 in the period (3 too high).
  So MAE is 1 in the window and 3 in the period, a change of +2; bias is +1 and +3.
- **X_1_HOBBIES** sells 20 every day except day 16, which is missing. The model
  forecasts 20 in the window (exact) and 18 in the period (2 too low).
  So MAE is 0 and 2, bias 0 and -2; the period has 6 actual days of 7, coverage
  6 / 7 = 0.857.
- The benchmark forecasts 99 everywhere; the tools leave it out unless asked.

In [ ]:
from typing import Any

from agentic_forecast_review.generator import forecast_row_id


def hand_actuals(
    series_id: str, value: int, skip: int | None = None
) -> list[ActualRow]:
    store, category = series_id.rsplit("_", 1)
    return [
        ActualRow(actual_row_id(series_id, day), series_id, store, category, day, value)
        for day in range(1, 22)
        if day != skip
    ]


def hand_forecasts(
    series_id: str, model: str, window: int, period: int
) -> list[ForecastRow]:
    return [
        ForecastRow(
            forecast_row_id(series_id, model, 7, day - 7),
            series_id,
            model,
            7,
            day,
            day - 7,
            window if day <= 14 else period,
        )
        for day in range(8, 22)
    ]


hand = Snapshot(
    seed=0,
    actuals=tuple(
        hand_actuals("X_1_FOODS", 10) + hand_actuals("X_1_HOBBIES", 20, skip=16)
    ),
    forecasts=tuple(
        hand_forecasts("X_1_FOODS", "model_v1", 11, 13)
        + hand_forecasts("X_1_HOBBIES", "model_v1", 20, 18)
        + hand_forecasts("X_1_FOODS", BENCHMARK_MODEL, 99, 99)
        + hand_forecasts("X_1_HOBBIES", BENCHMARK_MODEL, 99, 99)
    ),
)
window, period = DayRange(start_day=8, end_day=14), DayRange(start_day=15, end_day=21)

compare_runs matches the hand-worked numbers exactly.

In [ ]:
compared = compare_runs(
    hand,
    21,
    CompareRunsArgs(
        series=("X_1_FOODS", "X_1_HOBBIES"),
        metric=Metric.MAE,
        period=period,
        window=window,
    ),
).fields
for name, value in compared.items():
    print(name, value)
assert (compared["x_1_foods__mae_window"], compared["x_1_foods__mae_period"]) == (1, 3)
assert (compared["x_1_foods__bias_period"], compared["x_1_foods__mae_delta"]) == (3, 2)
assert (compared["x_1_hobbies__mae_period"], compared["x_1_hobbies__bias_period"]) == (
    2,
    -2,
)
assert compared["x_1_hobbies__n_actual_period"] == 6
assert compared["x_1_hobbies__model_period"] == "model_v1"

check_quality finds day 16 missing and the coverage of 0.857; inspect_series shows the
period day by day.

In [ ]:
quality = check_quality(
    hand, 21, CheckQualityArgs(series=("X_1_HOBBIES",), days=period)
)
print(quality.fields, quality.row_ids)
assert quality.fields["x_1_hobbies__missing_days"] == 1
assert quality.fields["x_1_hobbies__coverage"] == 0.857
assert quality.fields["x_1_hobbies__horizon_range"] == "8-14"
assert quality.row_ids == ("act__X_1_HOBBIES__d0016",)

inspected = inspect_series(
    hand, 21, InspectSeriesArgs(series_id="X_1_FOODS", days=period)
)
print(inspected.fields)
assert inspected.fields == {
    "n_days": 7,
    "mean_actual": 10,
    "mean_forecast": 13,
    "bias_last_7": 3,
    "bias_last_14": 3,
}

evidence_rows finds the rows that exist, and reports the one that does not.

In [ ]:
evidence = evidence_rows(
    hand,
    21,
    EvidenceRowsArgs(row_ids=("act__X_1_FOODS__d0015", "act__X_1_HOBBIES__d0016")),
)
print(evidence.fields, evidence.table)
assert evidence.fields == {"n_rows": 2, "n_found": 1}

context = resolve_context(hand, 21, ResolveContextArgs())
print(context.fields)
assert (
    context.fields["n_series"] == 2 and context.fields["metrics"] == "mae,signed_bias"
)

## The real cases, through the gateway

Cases 2-6 all ask about an error that looks worse, and in each it does. What differs is
the evidence behind it. In case 2 four actual days are missing; in case 6 the data is
complete and the bias has turned negative (the model under-forecasts), which makes 2 and
6 the matched pair. Case 1, the clean data, does not look worse at all.

In [ ]:
from agentic_forecast_review.cases import build_family
from agentic_forecast_review.gateway import Gateway

family = build_family()


def ask(number: int, request: dict[str, Any]) -> dict[str, FieldValue]:
    case = family.cases[number - 1]
    gateway = Gateway(case, family.snapshots[case.snapshot_id], TOOLS)
    result = gateway.call(request).result
    assert result.ok, result.error
    return result.fields


compare = {
    "tool": "compare_runs",
    "series": ["CA_2_FOODS"],
    "metric": "mae",
    "period": {"start_day": 169, "end_day": 196},
    "window": {"start_day": 141, "end_day": 168},
}
quality = {"tool": "check_quality", "series": ["CA_2_FOODS"], "days": compare["period"]}
for number in (1, 2, 3, 4, 5, 6):
    fields = ask(number, compare) | ask(number, quality)
    print(
        f"case {number}:",
        {
            name.removeprefix("ca_2_foods__"): fields[name]
            for name in (
                "ca_2_foods__mae_window",
                "ca_2_foods__mae_period",
                "ca_2_foods__model_window",
                "ca_2_foods__bias_period",
                "ca_2_foods__n_actual_period",
                "ca_2_foods__missing_days",
            )
        },
    )
assert ask(2, compare)["ca_2_foods__n_actual_period"] == 24
assert ask(6, quality)["ca_2_foods__missing_days"] == 0
assert (
    ask(6, compare)["ca_2_foods__bias_period"]
    < ask(1, compare)["ca_2_foods__bias_period"] - 5
)

The other planted conditions, each seen by the tool that should see it.

In [ ]:
both = {"start_day": 141, "end_day": 196}
dq_both = {**quality, "days": both}
duplicates = ask(3, quality)["ca_2_foods__duplicate_rows"]
models = (
    ask(4, compare)["ca_2_foods__model_window"],
    ask(4, compare)["ca_2_foods__model_period"],
)
horizons = ask(5, dq_both)["ca_2_foods__horizon_range"]
options = ask(8, {"tool": "resolve_context"})["metrics"]
print("case 3 duplicates:", duplicates)
print("case 4 models:", models)
print("case 5 horizons:", horizons)
print("case 8 metric options:", options)
assert duplicates == 3 and models == ("synthetic_v2", "synthetic_v3")
assert horizons == "1-7,22-28" and options == "mae,signed_bias"